In [11]:
!pip install konlpy

In [41]:
# [단계 1] 환경 초기화
import sys
# 1. 현재 노트북이 사용 중인 파이썬 경로 확인
print("현재 파이썬 실행 경로:", sys.executable)

# 2. 강제로 현재 파이썬 경로에 konlpy 설치 (이러면 100% 해결됩니다)
!{sys.executable} -m pip install konlpy
import os
import sqlite3
import pandas as pd
from konlpy.tag import Okt

# 경로 설정
project_root = "/Users/gangseongbin/Desktop/develop/python/AIProject"
if project_root not in sys.path:
    sys.path.append(project_root)

from Data.DataCleaner import DataCleaner


현재 파이썬 실행 경로: /Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [42]:
# [단계 2] 정제 및 DB 업데이트 함수
def run_final_cleaning():
    db_path = os.path.join(project_root, 'recipes.db')
    conn = sqlite3.connect(db_path)
    
    # 2. 새로운 정제 로직이 적용된 Cleaner 객체 생성
    cleaner = DataCleaner() 
    
    df_raw = pd.read_sql("SELECT materials FROM recipes", conn)
    
    # 3. 새로운 로직으로 완벽하게 새로 생성
    df_raw['pure_materials'] = df_raw['materials'].apply(cleaner.clean_ingredient)
    
    # 4. 기존 테이블을 완전히 삭제하고 새 데이터로만 구성된 테이블로 덮어쓰기
    df_raw.to_sql('recipes', conn, if_exists='replace', index=False)
    
    conn.close()
    print("✅ DB 테이블이 원본 데이터 기반으로 완전히 재구축되었습니다.")

In [43]:
# 실행
run_final_cleaning()

# 정제 결과 확인 (마지막 확인!)
conn = sqlite3.connect(os.path.join(project_root, 'recipes.db'))
df_final = pd.read_sql("SELECT pure_materials FROM recipes", conn)
print(df_final.head())
conn.close()


[알림] '고추'이(가) 자주 등장합니다. 사전에 등록하시겠습니까?

[알림] '가루'이(가) 자주 등장합니다. 사전에 등록하시겠습니까?

[알림] '약간'이(가) 자주 등장합니다. 사전에 등록하시겠습니까?

[알림] '튀김'이(가) 자주 등장합니다. 사전에 등록하시겠습니까?

[알림] '김치'이(가) 자주 등장합니다. 사전에 등록하시겠습니까?

[알림] '고춧가루'이(가) 자주 등장합니다. 사전에 등록하시겠습니까?

[알림] '떡볶이'이(가) 자주 등장합니다. 사전에 등록하시겠습니까?

[알림] '적당'이(가) 자주 등장합니다. 사전에 등록하시겠습니까?
✅ DB 테이블이 원본 데이터 기반으로 완전히 재구축되었습니다.
                                    pure_materials
0          대파, 당면, 느타리버섯, 소고기, 약간, 당근, 채소, 불고기, 양파
1  식초, 양념, 겨자, 올리브유, 후춧가루, 약간, 설탕, 채소, 간장, 참기름, 마늘
2               표고버섯, 당면, 소고기, 당근, 채소, 불고기, 양파, 피망
3        대파, 다시마, 기타, 손질, 고등어, 고추, 조림, 채소, 쌀뜨물, 양파
4                      대파, 오징어, 기타, 두부, 고추, 채소, 양파
